In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer

df = pd.read_csv('car_fuel_efficiency_2026.csv')

df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [2]:
print("Dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

Dataset shape: (10000, 11)

Missing values:
model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64


In [3]:
df = df.fillna(0)

print("Total missing values:", df.isnull().sum().sum())
print("Dataset shape:", df.shape)

Total missing values: 0
Dataset shape: (10000, 11)


In [4]:
df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=1
)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

print("Training:", df_train.shape)
print("Validation:", df_val.shape)
print("Testing:", df_test.shape)

Training: (6000, 11)
Validation: (2000, 11)
Testing: (2000, 11)


In [5]:
y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

print("Training target:", y_train.shape)
print("Validation target:", y_val.shape)
print("Testing target:", y_test.shape)
print("Remaining features:", df_train.shape[1])

Training target: (6000,)
Validation target: (2000,)
Testing target: (2000,)
Remaining features: 10


In [6]:
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')
test_dicts = df_test.to_dict(orient='records')

dv = DictVectorizer(sparse=True)

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)
X_test = dv.transform(test_dicts)

print("Training matrix:", X_train.shape)
print("Validation matrix:", X_val.shape)
print("Testing matrix:", X_test.shape)
print("Sparse matrix:", dv.sparse)

Training matrix: (6000, 16)
Validation matrix: (2000, 16)
Testing matrix: (2000, 16)
Sparse matrix: True


In [7]:
from sklearn.tree import DecisionTreeRegressor, export_text

dt = DecisionTreeRegressor(max_depth=1, random_state=1)
dt.fit(X_train, y_train)

print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



In [8]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

rf = RandomForestRegressor(
    n_estimators=10,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print("Validation RMSE:", rmse)
print("Rounded RMSE:", round(rmse, 3))

Validation RMSE: 1.833219735874562
Rounded RMSE: 1.833


In [9]:
n_estimators_values = [10, 50, 100, 150]

for n in n_estimators_values:
    rf = RandomForestRegressor(
        n_estimators=n,
        random_state=1,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, y_pred))

    print(f"n_estimators={n}: RMSE={rmse:.3f}")

n_estimators=10: RMSE=1.833
n_estimators=50: RMSE=1.774
n_estimators=100: RMSE=1.770
n_estimators=150: RMSE=1.769


In [10]:
max_depth_values = [10, 15, 20, 25]
n_estimators_values = [10, 50, 100, 150]

for depth in max_depth_values:
    rmse_scores = []

    for n in n_estimators_values:
        rf = RandomForestRegressor(
            n_estimators=n,
            max_depth=depth,
            random_state=1,
            n_jobs=-1
        )

        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        rmse_scores.append(rmse)

    mean_rmse = np.mean(rmse_scores)

    print(f"max_depth={depth}: Mean RMSE={mean_rmse:.3f}")

max_depth=10: Mean RMSE=1.754
max_depth=15: Mean RMSE=1.784
max_depth=20: Mean RMSE=1.787
max_depth=25: Mean RMSE=1.786


In [11]:
rf = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train)

feature_names = dv.get_feature_names_out()

importances = pd.DataFrame({
    'feature': feature_names,
    'importance': rf.feature_importances_
})

importances = importances.sort_values(
    by='importance',
    ascending=False
)

print(importances.to_string(index=False))

                     feature  importance
                  model_year    0.296107
              vehicle_weight    0.203313
          fuel_type=Gasoline    0.160712
                  horsepower    0.078271
         engine_displacement    0.061281
                acceleration    0.054998
                  origin=USA    0.042152
drivetrain=Front-wheel drive    0.020062
                   num_doors    0.019401
            fuel_type=Diesel    0.016571
            fuel_type=Hybrid    0.016272
  drivetrain=All-wheel drive    0.007951
               num_cylinders    0.007318
                 origin=Asia    0.006515
 drivetrain=Rear-wheel drive    0.005638
               origin=Europe    0.003436


In [12]:
import xgboost as xgb

features = list(dv.get_feature_names_out())

dtrain = xgb.DMatrix(
    X_train,
    label=y_train,
    feature_names=features
)

dval = xgb.DMatrix(
    X_val,
    label=y_val,
    feature_names=features
)

watchlist = [
    (dtrain, 'train'),
    (dval, 'val')
]

print("XGBoost version:", xgb.__version__)
print("Training rows:", dtrain.num_row())
print("Validation rows:", dval.num_row())
print("Number of features:", dtrain.num_col())

XGBoost version: 3.4.1
Training rows: 6000
Validation rows: 2000
Number of features: 16


In [13]:
xgb_params = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,
    'objective': 'reg:squarederror',
    'nthread': 8,
    'seed': 1,
    'verbosity': 1,
}

model_03 = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=100,
    evals=watchlist,
    verbose_eval=10
)

[0]	train-rmse:2.40653	val-rmse:2.46782
[10]	train-rmse:1.47125	val-rmse:1.73205
[20]	train-rmse:1.39368	val-rmse:1.73766
[30]	train-rmse:1.33243	val-rmse:1.74548
[40]	train-rmse:1.25326	val-rmse:1.76296
[50]	train-rmse:1.18645	val-rmse:1.77517
[60]	train-rmse:1.10265	val-rmse:1.79401
[70]	train-rmse:1.04604	val-rmse:1.80444
[80]	train-rmse:0.99487	val-rmse:1.81308
[90]	train-rmse:0.94986	val-rmse:1.82135
[99]	train-rmse:0.90106	val-rmse:1.82646


In [14]:
xgb_params_01 = {
    'eta': 0.1,
    'max_depth': 6,
    'min_child_weight': 1,
    'objective': 'reg:squarederror',
    'nthread': 8,
    'seed': 1,
    'verbosity': 1,
}

model_01 = xgb.train(
    xgb_params_01,
    dtrain,
    num_boost_round=100,
    evals=watchlist,
    verbose_eval=10
)

[0]	train-rmse:2.74739	val-rmse:2.78635
[10]	train-rmse:1.80592	val-rmse:1.93152
[20]	train-rmse:1.55847	val-rmse:1.75076
[30]	train-rmse:1.47544	val-rmse:1.71667
[40]	train-rmse:1.43074	val-rmse:1.71289
[50]	train-rmse:1.40368	val-rmse:1.71358
[60]	train-rmse:1.37884	val-rmse:1.71688
[70]	train-rmse:1.36283	val-rmse:1.71919
[80]	train-rmse:1.34578	val-rmse:1.72100
[90]	train-rmse:1.32546	val-rmse:1.72303
[99]	train-rmse:1.31192	val-rmse:1.72481
